In [1]:
import polars as pl
import os

In [2]:
DATASET_PATH = "/group/pmc021/amunif/epi-thesis/workflow/15_HepG2 from ENCODE/dataset"

In [3]:
# Function to read histone file and returns the data frame
def load_histone(filename):
    # Declare the intersect file schema
    # Define the schema for ensembl and gappedPeak dataset
    intersect_schema = pl.Schema({
        # The E118.bed columns

        "chromosome_name": pl.String,
        "start": pl.Int64,
        "end": pl.Int64,
        "gene_id": pl.String,
        "E118": pl.Float64,
        "strand": pl.Int64,
        "external_gene_name": pl.String,
        "start_position": pl.Int64,
        "end_position": pl.Int64,
        "tss": pl.Int64,
        
        # gappedPeak columns
        "chrom_p": pl.String,
        "chromStart_p": pl.Int64, 
        "chromEnd_p": pl.Int64, 
        "name_p": pl.String, 
        "score_p": pl.Float64, 
        "strand_p": pl.String,
        "thickStart": pl.Int64,
        "thickEnd": pl.Int64,
        "itemRgb": pl.Int64,
        "blockCount": pl.Int64,
        "blockSizes": pl.String,
        "blockStarts": pl.String,
        "signalValue": pl.Float64,
        "pValue": pl.Float64,
        "qValue": pl.Float64
    })

    # Open file
    histone_df = pl.read_csv(
            filename,
            separator="\t",
            has_header = False,
            schema = intersect_schema   
        )
    
    return histone_df

In [14]:
def create_empty_dataframe(histone_name):
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

In [16]:
def build_matrix(genes_df, histone_df, histone_name):
    # Build the dataframe with window
    genes_with_windows = genes_df.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_start')
    ]).explode('window_start')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_start') + 100).alias('window_end')
    ])

    # Join genes with histone data
    joined_df = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result_df = joined_df.filter(
        (pl.col('chromStart_p') < pl.col('window_end')) &
        (pl.col('chromEnd_p') > pl.col('window_start'))
    ).group_by(['gene_id', 'window_start'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_start'])

    # Find the gene without histone match
    genes_wo_histone = genes_with_windows.join(
        result_df,
        on=["gene_id", "window_start"],
        how="anti"
    )

    # Add the signalValue column so it can be merged
    genes_wo_histone = genes_wo_histone.with_columns(
        signalValue = pl.lit(0.0).cast(pl.Float64)
    )

    # Aggregate the genes without histone result
    genes_wo_histone = genes_wo_histone.group_by(['gene_id', 'window_start'], maintain_order=True).agg([
            pl.col('signalValue').mean().alias(histone_name)
        ]).sort(['gene_id', 'window_start'])

    # Merge both (results and genes without histone)
    result_df.extend(genes_wo_histone)

    # Sort the result dataframe by gene_id and window start for aggregation
    sorted_result_df = result_df.sort(['gene_id', 'window_start'])
    
    # Group by to make array of features
    matrix_df = (
        sorted_result_df
        .with_columns(pl.col(histone_name).fill_null(0))
        .group_by(['gene_id'], maintain_order=True)
        .agg(pl.col(histone_name))
        .sort('gene_id')
    )

    # Add the count and length of array feature for checking
    matrix_control_df = matrix_df.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    matrix_control_df = matrix_control_df.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    # Finally, return the gene_id with its histone features
    return matrix_control_df

In [17]:
# Getting histone in chunk
def get_histone_features(genes_df, histone_df, histone_name):
    
    genes_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes_df.iter_slices(n_rows=100)):
        if i % 100 == 0:
            print(f"Processing {histone_name}: {i*100}/{genes_df.height}")
        
        result = build_matrix(chunk, histone_df, histone_name)
        genes_w_histone.extend(result)

    print(f"Processing {histone_name} features finished.")
    
    return genes_w_histone

# Load the gene expression

In [6]:
col_names = [
    "chromosome_name",
    "start",
    "end",
    "gene_id",
    "E118",
    "strand",
    "external_gene_name",
    "start_position",
    "end_position",
    "tss"
]

E118_df = pl.read_csv(
    os.path.join(DATASET_PATH, "E118.bed"),
    separator = "\t",
    new_columns=col_names,
    has_header = False
)

In [7]:
E118_df

chromosome_name,start,end,gene_id,E118,strand,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",49.983,-1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.0,1,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",62.811,-1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",2.646,-1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",3.373,1,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",1.842,-1,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.0,-1,"""RP11-812E19.9""",33647044,33647696,33647696


# Load the histone dataset

In [8]:
# Load the histone dataset
H3K4me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K4me3.bed'))
H3K9ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K9ac.bed'))
H3K9me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K9me3.bed'))
H3K27ac_df = load_histone(os.path.join(DATASET_PATH, 'H3K27ac.bed'))
H3K27me3_df = load_histone(os.path.join(DATASET_PATH, 'H3K27me3.bed'))

In [15]:
H3K4me3_df

chromosome_name,start,end,gene_id,E118,strand,external_gene_name,start_position,end_position,tss,chrom_p,chromStart_p,chromEnd_p,name_p,score_p,strand_p,thickStart,thickEnd,itemRgb,blockCount,blockSizes,blockStarts,signalValue,pValue,qValue
str,i64,i64,str,f64,i64,str,i64,i64,i64,str,i64,i64,str,f64,str,i64,i64,i64,i64,str,str,f64,f64,f64
"""chrX""",99889988,99899988,"""ENSG00000000003""",49.983,-1,"""TSPAN6""",99883667,99894988,99894988,"""chrX""",99890988,99891866,"""H3K4me3_peak_44654""",281.0,""".""",99890988,99891866,0,3,"""1,811,1""","""0,62,877""",15.2192,30.4516,28.1466
"""chr20""",49570092,49580092,"""ENSG00000000419""",62.811,-1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49573333,49573513,"""H3K4me3_peak_28331""",22.0,""".""",49573333,49573513,0,2,"""1,1""","""0,179""",3.19282,4.09431,2.22335
"""chr20""",49570092,49580092,"""ENSG00000000419""",62.811,-1,"""DPM1""",49551404,49575092,49575092,"""chr20""",49573692,49576800,"""H3K4me3_peak_28332""",1056.0,""".""",49573692,49576800,0,3,"""170,2808,1""","""0,247,3107""",30.4445,108.934,105.637
"""chr1""",169858408,169868408,"""ENSG00000000457""",2.646,-1,"""SCYL3""",169818772,169863408,169863408,"""chr1""",169861388,169863941,"""H3K4me3_peak_3323""",309.0,""".""",169861388,169863941,0,4,"""1,186,1722,1""","""0,372,765,2552""",14.4684,33.2762,30.9914
"""chr1""",169858408,169868408,"""ENSG00000000457""",2.646,-1,"""SCYL3""",169818772,169863408,169863408,"""chr1""",169866112,169866295,"""H3K4me3_peak_3324""",21.0,""".""",169866112,169866295,0,2,"""1,1""","""0,182""",3.49716,4.01788,2.14846
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""chr15""",69678531,69688531,"""ENSG00000259645""",0.0,1,"""RP11-253M7.6""",69683531,69684488,69683531,"""chr15""",69687022,69687586,"""H3K4me3_peak_13814""",22.0,""".""",69687022,69687586,0,2,"""1,1""","""0,563""",3.55932,4.13522,2.26221
"""chr15""",102280913,102290913,"""ENSG00000259658""",1.842,-1,"""RP11-89K11.1""",102277302,102285913,102285913,"""chr15""",102285570,102285804,"""H3K4me3_peak_14503""",122.0,""".""",102285570,102285804,0,3,"""1,224,1""","""0,8,233""",8.74987,14.4097,12.2906
"""chr14""",103584344,103594344,"""ENSG00000259717""",0.0,-1,"""LINC00677""",103587184,103589344,103589344,"""chr14""",103584673,103584884,"""H3K4me3_peak_12893""",31.0,""".""",103584673,103584884,0,3,"""1,170,1""","""0,24,210""",4.11968,5.06932,3.15087


In [18]:
# Build the matrix
genes_w_H3K4me3_df = get_histone_features(E118_df, H3K4me3_df, 'H3K4me3')
genes_w_H3K9ac_df = get_histone_features(E118_df, H3K9ac_df, 'H3K9ac')
genes_w_H3K9me3_df = get_histone_features(E118_df, H3K9me3_df, 'H3K9me3')
genes_w_H3K27ac_df = get_histone_features(E118_df, H3K27ac_df, 'H3K27ac')
genes_w_H3K27me3_df = get_histone_features(E118_df, H3K27me3_df, 'H3K27me3')

Processing H3K4me3: 0/19645
Processing H3K4me3: 10000/19645
Processing H3K4me3 features finished.
Processing H3K9ac: 0/19645
Processing H3K9ac: 10000/19645
Processing H3K9ac features finished.
Processing H3K9me3: 0/19645
Processing H3K9me3: 10000/19645
Processing H3K9me3 features finished.
Processing H3K27ac: 0/19645
Processing H3K27ac: 10000/19645
Processing H3K27ac features finished.
Processing H3K27me3: 0/19645
Processing H3K27me3: 10000/19645
Processing H3K27me3 features finished.


# Checking the generated features

In [19]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_wc') > 0).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32
"""ENSG00000101040""","[14.1102, 14.1102, … 14.1102]",100,100
"""ENSG00000114315""","[9.61892, 9.61892, … 9.61892]",100,100
"""ENSG00000120129""","[14.7424, 14.7424, … 14.7424]",100,100
"""ENSG00000182095""","[20.2331, 20.2331, … 20.2331]",100,100
"""ENSG00000089006""","[10.971, 10.971, … 4.2722]",99,100
…,…,…,…
"""ENSG00000234965""","[0.0, 0.0, … 3.09679]",1,100
"""ENSG00000235718""","[3.02624, 0.0, … 0.0]",1,100
"""ENSG00000241186""","[0.0, 0.0, … 3.33662]",1,100


In [20]:
genes_w_H3K9ac_df.filter(pl.col('H3K9ac_len') < 100).sort(['H3K9ac_wc'], descending=True)

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len
str,list[f64],u32,u32


In [21]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_wc') > 0).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000185177""","[5.19098, 5.19098, … 5.19098]",100,100
"""ENSG00000187005""","[4.52623, 4.52623, … 5.29128]",100,100
"""ENSG00000196263""","[3.86704, 3.86704, … 3.86704]",100,100
"""ENSG00000214581""","[6.29437, 6.29437, … 6.29437]",100,100
"""ENSG00000253314""","[4.87992, 4.87992, … 4.87992]",100,100
…,…,…,…
"""ENSG00000189058""","[0.0, 0.0, … 2.61306]",1,100
"""ENSG00000196659""","[0.0, 0.0, … 2.09322]",1,100
"""ENSG00000204979""","[0.0, 0.0, … 2.65507]",1,100


In [22]:
genes_w_H3K9me3_df.filter(pl.col('H3K9me3_len') < 100).sort(['H3K9me3_wc'], descending=True)

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32


In [23]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_wc') > 0).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000182095""","[25.7556, 25.7556, … 25.7556]",100,100
"""ENSG00000099624""","[0.0, 0.0, … 16.1295]",98,100
"""ENSG00000006016""","[0.0, 0.0, … 13.009]",94,100
"""ENSG00000117394""","[6.04266, 6.04266, … 0.0]",94,100
"""ENSG00000167470""","[16.1295, 16.1295, … 17.9676]",94,100
…,…,…,…
"""ENSG00000181781""","[0.0, 0.0, … 8.33641]",1,100
"""ENSG00000182557""","[0.0, 0.0, … 3.3384]",1,100
"""ENSG00000198844""","[0.0, 0.0, … 3.71856]",1,100


In [24]:
genes_w_H3K4me3_df.filter(pl.col('H3K4me3_len') < 100).sort(['H3K4me3_wc'], descending=True)

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32


In [25]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_wc') > 0).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32
"""ENSG00000104760""","[23.9515, 23.9515, … 23.9515]",100,100
"""ENSG00000101040""","[24.5305, 24.5305, … 0.0]",99,100
"""ENSG00000112245""","[23.6256, 23.6256, … 18.3577]",99,100
"""ENSG00000182095""","[11.2074, 11.2074, … 19.781]",99,100
"""ENSG00000047457""","[17.8008, 17.8008, … 0.0]",98,100
…,…,…,…
"""ENSG00000186912""","[3.50226, 0.0, … 0.0]",1,100
"""ENSG00000188163""","[5.40588, 0.0, … 0.0]",1,100
"""ENSG00000212997""","[0.0, 0.0, … 2.79434]",1,100


In [26]:
genes_w_H3K27ac_df.filter(pl.col('H3K27ac_len') < 100).sort(['H3K27ac_wc'], descending=True)

gene_id,H3K27ac,H3K27ac_wc,H3K27ac_len
str,list[f64],u32,u32


In [27]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_wc') > 0).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000197616""","[4.04405, 4.04405, … 4.04405]",100,100
"""ENSG00000132130""","[4.74379, 4.74379, … 3.45311]",98,100
"""ENSG00000135903""","[3.69084, 3.69084, … 4.58607]",98,100
"""ENSG00000163081""","[3.69084, 3.69084, … 4.58607]",98,100
"""ENSG00000163499""","[0.0, 5.09242, … 4.45346]",98,100
…,…,…,…
"""ENSG00000197889""","[3.27736, 0.0, … 0.0]",1,100
"""ENSG00000206384""","[0.0, 0.0, … 3.16381]",1,100
"""ENSG00000227392""","[3.09319, 0.0, … 0.0]",1,100


In [28]:
genes_w_H3K27me3_df.filter(pl.col('H3K27me3_len') < 100).sort(['H3K27me3_wc'], descending=True)

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32


# Join all histones into single data frame

In [29]:
# Join all histone into single dataframe
genes_histone_df = genes_w_H3K9ac_df \
                    .join(genes_w_H3K9me3_df, on='gene_id') \
                    .join(genes_w_H3K4me3_df, on='gene_id') \
                    .join(genes_w_H3K27ac_df, on='gene_id') \
                    .join(genes_w_H3K27me3_df, on='gene_id')

In [30]:
genes_histone_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",15,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",11,100,"[0.0, 0.0, … 0.0]",4,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",40,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",25,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",15,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 3.25704]",3,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",3,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[3.2037, 3.2037, … 3.45265]",93,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


In [31]:
genes_values_df = E118_df.select(['gene_id', 'E118'])
genes_values_df

gene_id,E118
str,f64
"""ENSG00000000003""",49.983
"""ENSG00000000005""",0.0
"""ENSG00000000419""",62.811
"""ENSG00000000457""",2.646
"""ENSG00000000460""",3.373
…,…
"""ENSG00000259658""",1.842
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.0


In [32]:
genes_histone_values_df = genes_histone_df.join(
    genes_values_df,
    on = 'gene_id'
)

In [33]:
genes_histone_values_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E118
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",15,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",11,100,"[0.0, 0.0, … 0.0]",4,100,49.983
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",40,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",25,100,62.811
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",15,100,2.646
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 3.25704]",3,100,3.373
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",3,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,1.842
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[3.2037, 3.2037, … 3.45265]",93,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0


In [34]:
# Save to parquet
genes_histone_values_df.write_parquet(os.path.join(DATASET_PATH, 'E118_exp_histones.parquet'))

In [35]:
test_df = pl.read_parquet(os.path.join(DATASET_PATH, 'E118_exp_histones.parquet'))
test_df

gene_id,H3K9ac,H3K9ac_wc,H3K9ac_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K27ac,H3K27ac_wc,H3K27ac_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E118
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",15,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",9,100,"[0.0, 0.0, … 0.0]",11,100,"[0.0, 0.0, … 0.0]",4,100,49.983
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",40,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",25,100,62.811
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",27,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",29,100,"[0.0, 0.0, … 0.0]",20,100,"[0.0, 0.0, … 0.0]",15,100,2.646
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 3.25704]",3,100,3.373
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",3,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,1.842
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[3.2037, 3.2037, … 3.45265]",93,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.0
